In [23]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [100]:
from google import genai
from google.genai import types
genai_client = genai.Client(
    api_key= os.environ.get("GENAI_API_KEY")
)

In [25]:
from openai import OpenAI
groq_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key= os.getenv("GROQ_API_KEY")
)

In [26]:
import requests
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get("https://datatalks.club/faq/json/courses.json")
courses_raw = response.json()
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [27]:
documents = []
url_prefix = "https://datatalks.club/faq"
for course in courses_raw:
    url = f"""{url_prefix}{course['path']}"""
    course_response = requests.get(url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)
    
print(len(documents))

1342


In [28]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [29]:
from minsearch import Index
index = Index(
    text_fields=["section","question","answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [30]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [31]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'When will the course be offered next?',
 'I missed the first homework - can I still get a certificate?']

In [32]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [33]:
search_results = search(question)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [34]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [35]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [39]:
def build_context(search_results):
    lines= []
    
    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc['question'])
        lines.append("A: " + doc["answer"])
        lines.append("")
        
    return '\n'.join(lines).strip()

In [40]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context = context
    )
    
    return prompt.strip()

In [41]:
prompt = build_prompt(question,search_results)
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [45]:
response = groq_client.responses.create(
    input=prompt,
    model="llama-3.3-70b-versatile"
)

In [58]:
response.output_text

'Yes, you can still join the course, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [60]:
response.usage.input_tokens

366

In [70]:
input_price = .44 / 1000000
output_price = .67 / 1000000

cost = (
    input_price * response.usage.input_tokens +
    output_price * response.usage.output_tokens 
)

print(f"{cost:.5f}$")

0.00018$


In [87]:
model = 'gemini-3.5-flash'

In [101]:
def llm(instructions, user_prompt, model=model):
    message_history = [
        {'role':'system','content': instructions},
        {'role':'user','content':user_prompt}
    ] 
    
    # response = groq_client.responses.create(
    #     model=model,
    #     input=message_history
    # )
    
    response = genai_client.models.generate_content(
        model="gemini-3.5-flash",
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=instructions
        )
    )

    return response.text

In [102]:
def rag(query, model=model):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [103]:
answer = rag("How do I get a certificate?")
print(answer)

Based on the provided context, here is how you can get a certificate:

1. **Finish the course with a "live" cohort:** Certificates are not awarded for the self-paced mode.
2. **Pass the Capstone project:** Passing the Capstone project is mandatory to get the certificate (homework is recommended but not mandatory).
3. **Peer-review projects:** You must peer-review 3 capstone projects after submitting your own. This can only be done during the active course run after the peer-review list is compiled.
4. **Update your official name:** To ensure your real name appears on the certificate (and not a temporary random name), go to "Edit Course Profile" and update the second field with your official name as it appears on your identification documents.
